# Домашняя бухгалтерия с нейросетью — Этап 1 (данные)
Блокнот для Google Colab (бесплатный CPU достаточен). Запустите **Runtime → Run all**.
Что делает: скачивает открытые датасеты, собирает `category_dataset`, выгружает позиции реальных чеков для ручной разметки и запускает проверки Этапа 1.

**Правила:** никаких предобученных моделей; зеркала датасетов не используем — если ссылка недоступна, блокнот просто сообщит об этом.

## 1. Код проекта
Вариант А — репозиторий на GitHub (укажите URL). Вариант Б — загрузите `home-budget.zip` в файлы Colab.

In [ ]:
import os, subprocess, zipfile
REPO_URL = ""          # например: "https://github.com/<user>/home-budget.git"
if REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, "home-budget"], check=True)
elif os.path.exists("home-budget.zip"):
    zipfile.ZipFile("home-budget.zip").extractall(".")
else:
    raise SystemExit("Укажите REPO_URL или загрузите home-budget.zip")
%cd home-budget

In [ ]:
!pip -q install duckdb pyarrow pandas scikit-learn pyyaml huggingface_hub pytest

## 2. Быстрая проверка кода

In [ ]:
!python -m pytest -q

## 3. Скачивание данных (D1, D3–D6) и извлечение русских названий
D1 весит несколько ГБ — это нормально. Ошибки по отдельным источникам не останавливают блокнот.

**Повторный запуск после обновления кода:** уже скачанные файлы пропускаются (`already there`), поэтому повторно выполняется только извлечение — это быстро. Новое извлечение дописывает в CSV колонки `lang` и `countries`, по ним шаг 4 отсеивает болгарские и украинские названия (метка языка, страна, слова и словарь, который шаг 4 выучивает по самим данным). Если пропустить повторное извлечение, шаг 4 всё равно работает (по буквам и словарю), просто менее точно.

In [ ]:
!python -m ml.scripts.01_download

## 4. Сборка датасета
Без скачанных данных можно сделать пробный прогон: `--seed-only`.

In [ ]:
!python -m ml.scripts.02_build_category_dataset

In [ ]:
# Что сделала очистка: сколько строк отброшено / перемещено и примеры. Просмотрите глазами.
import json
rep = json.load(open("ml/reports/off_guard_report.json", encoding="utf-8"))
print("файлы:", json.dumps(rep["files"], ensure_ascii=False))
for k, v in rep["actions"].items():
    print(f"{v['count']:7d}  {k}")
    for ex in v["examples"][:3]:
        print("           ", ex)
# слова, которые шаг 4 сам выучил как «не русские» (встречаются только в болгарских/украинских названиях)
words = rep.get("learned_foreign_words", [])
print(f"\nвыученные нерусские слова: {len(words)}; первые 60 (слово, сколько нерусских названий, сколько русских):")
print(words[:60])


## 5. Разметка реальных чеков (нужна для сравнения стиля)
Если `real_test_labeled.csv` (разметка реальных чеков) уже есть у вас на компьютере, загрузите его сюда — в архив он не входит, потому что содержит реальные данные. Сравнение стиля на следующем шаге делится на «товары» и «кафе» по этой разметке; без неё сравнивается всё вместе.

In [ ]:
import os, shutil
from google.colab import files
target = "ml/data/processed/real_test_labeled.csv"
if not os.path.exists(target):
    up = files.upload()   # выберите real_test_labeled.csv
    if "real_test_labeled.csv" in up:
        os.makedirs(os.path.dirname(target), exist_ok=True)
        shutil.move("real_test_labeled.csv", target)
print("real_test_labeled.csv:", os.path.exists(target))

## 5б. Сравнение стиля: наши «грязные» строки рядом с реальными из D6
Числа сравниваются отдельно для товаров (`goods`) и кафе (`meal`). У кафе доля строк с цифрой в начале и доля строк с цифрами показываются, но не оцениваются: 47% строк кафе в D6 начинаются с «1 » из-за формата одной кассы, повторять это в обучающих данных нельзя.

In [ ]:
!python -m ml.scripts.03_augment_receipt_style --n 25 --compare-d6

## 6. Реальный тест (D6)
Выгрузка позиций для ручной разметки. Скачайте `real_items_to_label.csv`, заполните колонку `subcategory`
(id из `data/taxonomy.yaml`, минимум 500 строк), загрузите обратно и выполните `finalize`.

In [ ]:
!python -m ml.scripts.04_label_real_test export

In [ ]:
from google.colab import files
files.download("ml/data/processed/real_items_to_label.csv")

In [ ]:
# после разметки: загрузите файл и запустите
# from google.colab import files; files.upload()
# !python -m ml.scripts.04_label_real_test finalize --labeled real_items_to_label.csv

## 7. Проверки Этапа 1

In [ ]:
!python -m ml.scripts.06_check_stage1 --show 20

In [ ]:
from google.colab import files
for f in ["ml/reports/class_distribution.json", "ml/reports/stage1_check.json", "ml/reports/off_map_samples.md", "ml/reports/raw_schemas.json", "ml/reports/off_guard_report.json", "ml/reports/style_compare.json"]:
    if os.path.exists(f): files.download(f)